# 01 Image Meta Tag Tutorial

This tutorial will guide you through the key components of using the Image Meta Tag package to:
* Tag images
* Create a database of images and associated tags. 
* Generate a webpage with selector menus to display those images using the image tags. 

The goal is to develop the following files in your `public_html` directory:
* A webpage which will display your images and selector dropdown menus. 
* A database file which will contain image locations and their associated tags. 
* Symlinks to the actual image locations (as its likely you don't want these in your home directory)

```text
├── webpage.html
├── databasefile.db
├── image1.png -> /data/scratch/user.name/image_meta_tag_test/image1.png
└── image2.png -> /data/scratch/user.name/image_meta_tag_test/image2.png
```

When you create the database you could do this with an existing directory of images as long as you have a method to create the tags for each one. You could do this by parsing the filename if it contained the tag data you require. Alternatively when you create the images you could either write out tags and image paths to the database, or export tag data when creating the charts. We do this in a project by creating json files for each image created that detail the image path and tags for that image as we have access to all chart data at that point. We can then read all the json files into the database. 

### Example
We are aiming to build a simple webpage in this tutorial with some selector menus and a single image display.

![Image Meta Tag Tutorial 2](images/image_meta_tag_tutorial_1.png)

### Setup
Check the `image_tag_environment.yml` file for environment requirements. 

In [10]:
# This tutorial uses a config file to set paths and avoid path exposure. 

import yaml
with open("config.yml", "r") as file:
    config = yaml.safe_load(file)

In [3]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import xarray as xr

from pathlib import Path

# Import the ImageMetaTag package:
import ImageMetaTag as imt

# Suppress warnings for demo purposes
import warnings
warnings.filterwarnings("ignore")

### Project paths
Define paths to be used in this project:
* Where to store the images
* Where to store the webpage elements (html, database and symlinks)

In [11]:
project_name = "image_meta_tag_tutorial_1"

# image_dir is used to save the actual image files.
image_dir = Path(config["image_dir_1"]) / project_name
image_dir.mkdir(parents=True, exist_ok=True)

# web_dir specifies a folder in the user's public_html folder 
# The public_html folder can be viewed at https://wwwspice/~user.name/folder/
home = Path(os.getenv("HOME"))
web_dir = home / "public_html" / project_name
web_dir.mkdir(parents=True, exist_ok=True)

# The database file should be saved in the web_dir with the images / or symlinks to images in another location. 
database_file_path = web_dir / "image_metadata.db"


In [12]:
# Ensure web-dir is empty - useful if re-running the notebook.
# Uncomment the function call to run.

def clear_web_dir(web_dir: Path):
    for item in web_dir.iterdir():
        if item.is_file() or item.is_symlink():
            item.unlink()

# clear_web_dir(web_dir)

### Data location
Define where the data is located we will use for this demo.

In [13]:
data_path = Path(config["data_path_1"])
file_list = sorted(data_path.glob("*.nc"))
files = [str(p) for p in file_list[:10]] # Just get the paths for the first 10 files for this demo

In [14]:
# Lets load one file to explore our data.
ds = xr.open_dataset(files[0])
ds

<xarray.Dataset> Size: 14MB
Dimensions:                      (grid_latitude: 810, grid_longitude: 621,
                                  grid_latitude_0: 811, grid_longitude_0: 621)
Coordinates:
  * grid_latitude                (grid_latitude) float32 3kB -3.771 ... 7.151
  * grid_longitude               (grid_longitude) float64 5kB 354.9 ... 363.3
    forecast_period              timedelta64[ns] 8B ...
    forecast_reference_time      datetime64[ns] 8B ...
    time                         datetime64[ns] 8B ...
    height                       float64 8B ...
    forecast_period_0            timedelta64[ns] 8B ...
    time_0                       datetime64[ns] 8B ...
  * grid_latitude_0              (grid_latitude_0) float32 3kB -3.777 ... 7.158
  * grid_longitude_0             (grid_longitude_0) float64 5kB 354.9 ... 363.3
    height_0                     float64 8B ...
Data variables:
    air_pressure_at_sea_level    (grid_latitude, grid_longitude) float32 2MB ...
    rotated_latitude_longitude   int32 4B ...
    air_temperature              (grid_latitude, grid_longitude) float32 2MB ...
    specific_humidity            (grid_latitude, grid_longitude) float32 2MB ...
    surface_altitude             (grid_latitude, grid_longitude) float32 2MB ...
    toa_outgoing_shortwave_flux  (grid_latitude, grid_longitude) float32 2MB ...
    x_wind                       (grid_latitude_0, grid_longitude_0) float32 2MB ...
    y_wind                       (grid_latitude_0, grid_longitude_0) float32 2MB ...
Attributes:
    source:       Data from Met Office Unified Model
    um_version:   10.6
    Conventions:  CF-1.7

### Helper function to plot charts

In [15]:
# This is a helper function to do the plotting and image creation. It returns a fig we will use with the image meta tag package. 
# We have extracted this functionality so you can focues on the Image Meta Tagging process in the next code cell.
def plot_rotated_pole_chart(ds, var_name, data, units, file_date, pole_longitude=177.5, pole_latitude=37.5):
    if data.ndim != 2:
        return None
    
    if "grid_latitude" in ds[var_name].dims and "grid_longitude" in ds[var_name].dims:
        lats = ds.coords["grid_latitude"].values
        lons = ds.coords["grid_longitude"].values
    elif "grid_latitude_0" in ds[var_name].dims and "grid_longitude_0" in ds[var_name].dims:
        lats = ds.coords["grid_latitude_0"].values
        lons = ds.coords["grid_longitude_0"].values
    else:
        return None

    lon2d, lat2d = np.meshgrid(lons, lats)
    rotated_pole = ccrs.RotatedPole(pole_longitude=pole_longitude, pole_latitude=pole_latitude)
    fig = plt.figure(figsize=(6, 8))
    ax = plt.axes(projection=rotated_pole)
    ax.set_title(f"{var_name} at {file_date}")
    ax.set_extent([np.min(lons), np.max(lons), np.min(lats), np.max(lats)], rotated_pole)
    ax.coastlines()
    mesh = ax.pcolormesh(lon2d, lat2d, data, transform=rotated_pole)
    plt.colorbar(mesh, ax=ax, orientation="vertical", label=units)
    return fig

### Produce images, tag them, create database and image symlinks

The following code block will demonstrate how to:
* Loop through your data files (1 per datetime in this case).
* Loop through the variables in a single file. 
* For each variable in each file:
    * Create a plot
    * Create tags for that plot
    * Save the plot and tags as an image using the Image Meta Tag .savefig() method.
    * Save the image location and tags to a database file using the Image Meta Tag .write_img_to_dbfile() method. 
    * Create a symlink to the image location. 

In [17]:
# Loop over our files and create some plots with image meta tags.
for file in files:
    print("Processing file:", Path(file).name)
    ds = xr.open_dataset(file)

    # Extract file-level metadata for tags. 
    time_val = pd.Timestamp(ds.coords["time"].values)
    file_date = str(time_val.date())
    source = ds.attrs.get("source") or "unknown-source"
    forecast_lead_time = ds.attrs.get("forecast_lead_time") or "000"
    forecast_reference_time = ds.attrs.get("forecast_reference_time") or str(time_val)

    # Loop through the variables in the dataset. 
    for var_name in ds.data_vars:
        print("\r", ' ' * 60, "\rProcessing:", var_name, "\r",  end='')
        
        # Extract variable-level metadata for tags.
        var = ds[var_name]
        units = var.attrs.get("units") or "unknown-units"
        
        # Use our helper function above to create and return a chart in the form of a plt.fig. 
        data = var.values
        fig = plot_rotated_pole_chart(ds, var_name, data, units, file_date)

        # Build the image filename and path
        filename = f"{var_name}_{file_date}.png"
        file_path = os.path.join(image_dir, filename)
        
        ### CREATE A DICTIONARY OF TAGS FOR EACH IMAGE ###
        # This is used to tag images, build the database and information for webpage selectors.
        img_tags = {
                "variable": var_name,
                "units": units,
                "date": file_date,
                "forecast_lead_time": forecast_lead_time,
                "forecast_reference_time": forecast_reference_time,
                "source": source,
            }

        ### USE THE IMAGE META TAG .savefig() METHOD TO TAG IMAGES ###
        # You may not need to do this depending on your use case.
        # It is possible to skip tagging the images and just write to the database directly (below).
        imt.savefig(file_path, fig=fig, img_tags=img_tags)


        ### CREATE AND WRITE DIRECTLY TO A DATABASE FILE USING THE IMAGE META TAG .write_img_to_dbfile() METHOD ###
        # This is useful if you have pre-existing images or do not want to tag the images directly.
        # We write the same tags as above to the database.   
        # The database file should be saved in a directory in your ~/user.name/public_html folder with the webpage files and symlinks.  
        imt.db.write_img_to_dbfile(database_file_path, filename, img_tags)
        

        ### CREATE A SYMLINK TO THE IMAGE IN THE WEB DIRECTORY ###
        # Create symlink path in web_dir
        symlink_path = web_dir / filename
        
        # Try to create the symlink
        try:
            if symlink_path.exists() or symlink_path.is_symlink():
                symlink_path.unlink()  # Remove existing file or symlink
            os.symlink(file_path, symlink_path)
        except Exception as e:
            print(f"Could not link {file_path} -> {symlink_path}: {e}")

        plt.close(fig) # Close the fig now we have saved the images. 
            
print("\r", ' ' * 60, "\rProcessing completed", end='')

Processing file: prods_op_ukv_20180101_03_000.nc
Processing file: prods_op_ukv_20180101_09_000.nc              
Processing file: prods_op_ukv_20180101_15_000.nc              
Processing file: prods_op_ukv_20180101_21_000.nc              
Processing file: prods_op_ukv_20180102_03_000.nc              
Processing file: prods_op_ukv_20180102_09_000.nc              
Processing file: prods_op_ukv_20180102_15_000.nc              
Processing file: prods_op_ukv_20180102_21_000.nc              
Processing file: prods_op_ukv_20180103_03_000.nc              
Processing file: prods_op_ukv_20180103_09_000.nc              
Processing completed                                          

### Read image metadata with a standard package
If you tagged your images in addition to creating the database you can read the image meta-data with a standard python image package like this:

In [18]:
from PIL import Image

img = Image.open(f"{image_dir}/air_temperature_2018-01-01.png")
print(img.info)

{'Software': 'Matplotlib version3.10.6, https://matplotlib.org/', 'variable': 'air_temperature', 'units': 'K', 'date': '2018-01-01', 'forecast_lead_time': '000', 'forecast_reference_time': '2018-01-01 21:00:00', 'source': 'Data from Met Office Unified Model'}


### Read image-meta-tag database file with image-meta-tag method
However, it will be more useful for the purpose of creating the webpage to read the meta-data from the database file like this:

In [19]:
# Read the database file into a variable.
imt_db = imt.db.read(database_file_path)

# Returns a tuple of file_paths and tags dictionary
img_file_paths = imt_db[0]
img_tags_dict = imt_db[1]

In [20]:
img_tags_dict

{'air_pressure_at_sea_level_2018-01-01.png': {'variable': 'air_pressure_at_sea_level',
  'units': 'Pa',
  'date': '2018-01-01',
  'forecast_lead_time': '000',
  'forecast_reference_time': '2018-01-01 03:00:00',
  'source': 'Data from Met Office Unified Model'},
 'rotated_latitude_longitude_2018-01-01.png': {'variable': 'rotated_latitude_longitude',
  'units': 'unknown-units',
  'date': '2018-01-01',
  'forecast_lead_time': '000',
  'forecast_reference_time': '2018-01-01 03:00:00',
  'source': 'Data from Met Office Unified Model'},
 'air_temperature_2018-01-01.png': {'variable': 'air_temperature',
  'units': 'K',
  'date': '2018-01-01',
  'forecast_lead_time': '000',
  'forecast_reference_time': '2018-01-01 03:00:00',
  'source': 'Data from Met Office Unified Model'},
 'specific_humidity_2018-01-01.png': {'variable': 'specific_humidity',
  'units': '1',
  'date': '2018-01-01',
  'forecast_lead_time': '000',
  'forecast_reference_time': '2018-01-01 03:00:00',
  'source': 'Data from Met O

### Create an ImageDict object required for webpage creation

In [ ]:
# Extract the image tag names from the image_tags_dict - useful if you wish to do this automatically / dynamically. 
img_tag_names = list(next(iter(img_tags_dict.values())).keys())

# Or, we can control the webpage dropdown menu visibility and order by manually specifying them.
img_tag_names: list = ["variable", "date", "forecast_lead_time", "forecast_reference_time", "source", "units"]

# Data for selector menu headings: 
# Map the tags to full names for the drop-down menu headings. 
sel_full_names_dict = {'variable': 'Variable',
                       'date': 'Date',
                       'forecast_lead_time': 'Lead Time',
                       'forecast_reference_time': 'Analysis Time',
                       'source': 'Source',
                       'units': 'Units'
                       }
# Map the tags to drop-down menu heading widths.
sel_widths_dict = {'variable': '240px',
                'date': '140px',
                'forecast_lead_time': '140px',
                'forecast_reference_time': '140px',
                'source': '240px',
                'units': '120px'
                }

selector_level_names: list = [sel_full_names_dict[x] for x in img_tag_names]
selector_widths: list = [sel_widths_dict[x] for x in img_tag_names]

In [22]:
# Parameters to create 'Step back' / 'Step forward' buttons.
# This enables the user to easily navigate through images based on a specific selector menu.

animation_direction = +1    # +1 to move forwards, -1 to move backwards. 
selector_animated = 1       # Choose which selector menu to operate on (1 = date in this case).

In [23]:
# Create the ImageDict for the webpage

img_dict = None

for img_file, img_info in img_tags_dict.items():
    tmp_dict = imt.dict_heirachy_from_list(img_info, img_file, img_tag_names)

    if not img_dict:
        img_dict = imt.ImageDict(tmp_dict,
                                 selector_animated=selector_animated,       # Which selector menu to animate.
                                 animation_direction=animation_direction,   # Animation direction
                                 level_names=selector_level_names,  # Add level_names to the img_dict to use as selector menu headers. 
                                 selector_widths=selector_widths)   # First database entry used to create the database.
    else:
        img_dict.append(imt.ImageDict(tmp_dict, 
                                      level_names=selector_level_names)) # Now append entries with level names for selector menu headers.

In [24]:
print(img_dict)

ImageMetaTag ImageDict:
  y_wind:
    2018-01-01:
      000:
        2018-01-01 03:00:00:
          Data from Met Office Unified Model:
            m s-1:
              y_wind_2018-01-01.png
    2018-01-03:
      000:
        2018-01-03 03:00:00:
          Data from Met Office Unified Model:
            m s-1:
              y_wind_2018-01-03.png
    2018-01-02:
      000:
        2018-01-02 03:00:00:
          Data from Met Office Unified Model:
            m s-1:
              y_wind_2018-01-02.png
  surface_altitude:
    2018-01-01:
      000:
        2018-01-01 03:00:00:
          Data from Met Office Unified Model:
            m:
              surface_altitude_2018-01-01.png
    2018-01-03:
      000:
        2018-01-03 03:00:00:
          Data from Met Office Unified Model:
            m:
              surface_altitude_2018-01-03.png
    2018-01-02:
      000:
        2018-01-02 03:00:00:
          Data from Met Office Unified Model:
            m:
              surface_altitude_2

### Create header and footer html for the webpage

In [ ]:
# Define webpage header and footer content:

margin_horizontal = 5
margin_vertical = 20

webpage_header = f'''
<div id='preamble' style='margin: {margin_vertical}px {margin_horizontal}px'>
    <h1>Image Meta Tag Tutorial 1</h1>
</div>
'''

webpage_footer = f'''
<div id='postamble' style='margin: {margin_vertical}px {margin_horizontal}px'>
  <h3>Footer area</h3>
</div>
'''

### Create the webpage

In [ ]:
out_page = web_dir / "image_meta_tag_tutorial_2.html"

web_out = {}
web_out[out_page] = imt.webpage.write_full_page(img_dict, out_page,
                                    'Test ImageDict webpage',
                                    preamble=webpage_header,
                                    postamble=webpage_footer,
                                    verbose=False, 
                                    only_show_rel_url=True,
                                    write_intmed_tmpfile=True,
                                    show_selector_names=True,       # Show the 'level_names' as selector menu headers. 
                                    show_singleton_selectors=True   # Show HTML selector menu even if just 1 value.
                                    )

print(f"Webpage available at: https://wwwspice/~user.name/public_html/ Look for the html file in your web_dir")

Webpage available at: https://wwwspice/~user.name/public_html/ Look for the html file in your web_dir


* Check your wwwspice/~user.name/public_html/ directory and look for your web_dir project name directory. Find the .html file in the project directory and open it! 

* Tutorial 2 demonstrates how to create a website with multiple charts. 